# **Maestría en Inteligencia Artificial Aplicada (MNA) - Tecnológico de Monterrey**
### **TC 4034 -  Análisis de Grandes Volúmenes de Datos**
----

### **Profesor:** 	Dr. Iván Olmos Pineda

### **Proyecto: Lectura, Escritura, Archivos de Big Data PySpark**

------

###**Team 14**

* Mariana Paola De los Cobos Kingston (**A01796922**)
* Gerardo Tenorio Castillo (**A01139576**)
* Óscar Benjamín Zacarías Villegas (**A01797160**)

----

## 1. Descripción de la Base de Datos

Así como se detalló en la primeera entrega, la base de datos proviene de los [SEC Financial Statement Data Sets](https://www.sec.gov/dera/data/financial-statement-data-sets.html),
publicados trimestralmente por la *U.S. Securities and Exchange Commission*. Contiene información financiera estructurada y estandarizada
reportada por miles de empresas públicas en Estados Unidos.

El dataset se compone de **cuatro tablas tabulares** (archivos `.txt` delimitados por tabulador) por cada trimestre:

| Tabla | Descripción | Registros |
|-------|-------------|----------:|
| `sub.txt` | Metadatos del reporte (empresa, formulario, fechas) | ~66,029 |
| `num.txt` | Valores numéricos reportados por concepto financiero | ~36,093,438 |
| `pre.txt` | Presentación visual de los conceptos en el reporte | ~7,512,071 |
| `tag.txt` | Definiciones de los tags/conceptos contables | ~823,510 |

Tras consolidar las cuatro tablas mediante sus claves de unión (`adsh`, `tag`, `version`), se obtiene un DataFrame unificado (`full_df`)
con **~50.6 millones de registros** y **59 columnas**.

El periodo cubierto abarca desde **2023 hasta 2026**, con reportes trimestrales (10-Q) y anuales (10-K), lo que permite análisis de tendencias temporales y comparaciones sectoriales.

## Relevancia del Dataset para el Tema de Investigación



El dataset de la SEC es altamente relevante para este proyecto porque concentra la información financiera estructurada y estandarizada reportada por miles de empresas públicas ante el regulador federal de valores de Estados Unidos.

En particular, permite abordar los siguientes ejes del tema de investigación:

- **Rentabilidad**: A través de tags como `NetIncomeLoss`, `GrossProfit` y `OperatingIncomeLoss` presentes en la tabla `num.txt`, es posible calcular métricas como ROE y ROA para comparar el desempeño entre empresas y sectores.

- **Liquidez**: Variables como `AssetsCurrent`, `LiabilitiesCurrent` y `CashAndCashEquivalentsAtCarryingValue` permiten construir razones de liquidez (razón circulante, prueba ácida).

- **Endeudamiento**: Tags como `LongTermDebt`, `Liabilities` y `StockholdersEquity` habilitan el análisis de apalancamiento y estructura de deuda.

- **Estructura de capital**: La combinación de datos de deuda, capital y utilidades retenidas permite analizar cómo las empresas financian sus operaciones.

Al cubrir múltiples periodos trimestrales y anuales (desde 2023 hasta 2026), el dataset facilita el análisis de tendencias temporales y la comparación del comportamiento financiero bajo distintas condiciones económicas.

## 2. Carga de Datos - PySpark

Se incluye la opción de trabajar con Spark a través de Google Colab.

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [ ]:
!ls

In [ ]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, BooleanType, DateType, DecimalType, IntegerType, StringType, Row

spark = SparkSession.builder \
    .appName("SEC") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "48") \
    .getOrCreate()

spark

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 10)
spark.conf.set("spark.sql.ansi.enabled", "false")

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# base_path = r"C:/Users/ozaca/Documents/MNA/Big Data/SEC Financial Statement Data Sets"
base_path = r"C:\Users\getec\Documents\MNA\BigData\Proyecto\Entregable1\SEC Financial Statement Data Sets\SEC Financial Statement Data Sets"
#base_path = "/content/drive/MyDrive/Proyecto Big Data/SEC"

### Lectura de archivos tabulares

In [ ]:
# Generamos la lista de rutas explícitas con Python
carpetas = [os.path.join(base_path, f, "sub.txt")
            for f in os.listdir(base_path)
            if os.path.isdir(os.path.join(base_path, f))]

# Leemos pasando la lista directamente
sub_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "false") \
    .csv(carpetas) \
    .dropDuplicates()

print(f"Cargadas {len(carpetas)} carpetas.")

Cargadas 10 carpetas.


In [ ]:
print(f"Número de registros de tabla sub: {sub_df.count():,}")
sub_df.printSchema()
sub_df.show(10)

Número de registros de tabla sub: 66,029
root
 |-- adsh: string (nullable = true)
 |-- cik: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sic: string (nullable = true)
 |-- countryba: string (nullable = true)
 |-- stprba: string (nullable = true)
 |-- cityba: string (nullable = true)
 |-- zipba: string (nullable = true)
 |-- bas1: string (nullable = true)
 |-- bas2: string (nullable = true)
 |-- baph: string (nullable = true)
 |-- countryma: string (nullable = true)
 |-- stprma: string (nullable = true)
 |-- cityma: string (nullable = true)
 |-- zipma: string (nullable = true)
 |-- mas1: string (nullable = true)
 |-- mas2: string (nullable = true)
 |-- countryinc: string (nullable = true)
 |-- stprinc: string (nullable = true)
 |-- ein: string (nullable = true)
 |-- former: string (nullable = true)
 |-- changed: string (nullable = true)
 |-- afs: string (nullable = true)
 |-- wksi: string (nullable = true)
 |-- fye: string (nullable = true)
 |-- form: string (nullab

In [ ]:
# Generamos la lista de rutas explícitas con Python
carpetas = [os.path.join(base_path, f, "num.txt")
            for f in os.listdir(base_path)
            if os.path.isdir(os.path.join(base_path, f))]

# Leemos pasando la lista directamente
num_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "false") \
    .csv(carpetas) \

print(f"Cargadas {len(carpetas)} carpetas.")

Cargadas 10 carpetas.


In [ ]:
print(f"Número de registros de tabla num: {num_df.count():,}")
num_df.printSchema()
num_df.show(10)

Número de registros de tabla num: 36,093,438
root
 |-- adsh: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- version: string (nullable = true)
 |-- ddate: string (nullable = true)
 |-- qtrs: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- segments: string (nullable = true)
 |-- coreg: string (nullable = true)
 |-- value: string (nullable = true)
 |-- footnote: string (nullable = true)

+--------------------+--------------------+--------------------+--------+----+---+--------------------+-----+---------------+--------+
|                adsh|                 tag|             version|   ddate|qtrs|uom|            segments|coreg|          value|footnote|
+--------------------+--------------------+--------------------+--------+----+---+--------------------+-----+---------------+--------+
|0001437749-23-031921|AccumulatedOtherC...|        us-gaap/2023|20221231|   0|USD|                NULL| NULL| -21665000.0000|    NULL|
|0001493152-23-045550|AdjustmentsForD

In [ ]:
# Generamos la lista de rutas explícitas con Python
carpetas = [os.path.join(base_path, f, "pre.txt")
            for f in os.listdir(base_path)
            if os.path.isdir(os.path.join(base_path, f))]

# Leemos pasando la lista directamente
pre_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "false") \
    .csv(carpetas)

print(f"Cargadas {len(carpetas)} carpetas.")

Cargadas 10 carpetas.


In [ ]:
print(f"Número de registros de tabla pre: {pre_df.count():,}")
pre_df.printSchema()
pre_df.show(10)

Número de registros de tabla pre: 7,512,071
root
 |-- adsh: string (nullable = true)
 |-- report: string (nullable = true)
 |-- line: string (nullable = true)
 |-- stmt: string (nullable = true)
 |-- inpth: string (nullable = true)
 |-- rfile: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- version: string (nullable = true)
 |-- plabel: string (nullable = true)
 |-- negating: string (nullable = true)

+--------------------+------+----+----+-----+-----+--------------------+------------+--------------------+--------+
|                adsh|report|line|stmt|inpth|rfile|                 tag|     version|              plabel|negating|
+--------------------+------+----+----+-----+-----+--------------------+------------+--------------------+--------+
|0000002178-23-000091|     2|   9|  BS|    0|    H|CashAndCashEquiva...|us-gaap/2023|Cash and cash equ...|       0|
|0000002178-23-000091|     2|  10|  BS|    0|    H|RestrictedCashCur...|us-gaap/2023|     Restricted cash|       0

In [ ]:
# Generamos la lista de rutas explícitas con Python
carpetas = [os.path.join(base_path, f, "tag.txt")
            for f in os.listdir(base_path)
            if os.path.isdir(os.path.join(base_path, f))]

# Leemos pasando la lista directamente
tag_df = spark.read \
    .option("header", "true") \
    .option("delimiter", "\t") \
    .option("inferSchema", "false") \
    .csv(carpetas) \
    .dropDuplicates()

print(f"Cargadas {len(carpetas)} carpetas.")

Cargadas 10 carpetas.


In [ ]:
print(f"Número de registros de tabla tag: {tag_df.count():,}")
tag_df.printSchema()
tag_df.show(10)

Número de registros de tabla tag: 823,510
root
 |-- tag: string (nullable = true)
 |-- version: string (nullable = true)
 |-- custom: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- datatype: string (nullable = true)
 |-- iord: string (nullable = true)
 |-- crdr: string (nullable = true)
 |-- tlabel: string (nullable = true)
 |-- doc: string (nullable = true)

+--------------------+------------+------+--------+--------+----+----+--------------------+--------------------+
|                 tag|     version|custom|abstract|datatype|iord|crdr|              tlabel|                 doc|
+--------------------+------------+------+--------+--------+----+----+--------------------+--------------------+
|AccountsPayableAn...|us-gaap/2022|     0|       0|monetary|   I|   C|Accounts Payable ...|Sum of the carryi...|
|AdjustmentsToAddi...|us-gaap/2022|     0|       0|monetary|   D|   C|APIC, Share-Based...|Amount of increas...|
|AllocatedShareBas...|us-gaap/2022|     0|       0

In [ ]:
#Número de registros por tabla
print(f"sub: {sub_df.count():,}")
print(f"num: {num_df.count():,}")
print(f"pre: {pre_df.count():,}")
print(f"tag: {tag_df.count():,}")

sub: 66,029
num: 36,093,438
pre: 7,512,071
tag: 823,510


## 3. Consolidación de la Tabla Final

En esta sección se integran las cuatro tablas del dataset SEC en una sola tabla consolidada llamada full_df,
combinando los valores financieros reportados con metadatos de la empresa, definiciones contables y contexto de presentación.

### Claves de Unión de tipo Left Join

| Join | Tabla Izquierda | Tabla Derecha | Clave(s) de Unión | Tipo | Descripción |
|------|----------------|---------------|-------------------|------|-------------|
| 1 | num | tag | tag, version | LEFT | Agrega definiciones y metadatos del concepto contable (tipo de dato, naturaleza débito/crédito, etiqueta legible, documentación) |
| 2 | num_tag | pre | adsh, tag, version | LEFT | Incorpora el contexto de presentación visual (estado financiero, línea, etiqueta preferida, nivel de jerarquía) |
| 3 | num_tag_pre | sub | adsh | LEFT | Añade metadatos de la empresa y del reporte (nombre, industria SIC, tipo de formulario, periodo y año fiscal, fechas de presentación) |

### Resultado: full_df

| Característica | Valor |
|----------------|-------|
| Registros totales | ~50,628,691 |
| Columnas totales | 59 |
| Origen num | adsh, tag, version, ddate, qtrs, uom, segments, coreg, value, footnote |
| Origen tag | custom, abstract, datatype, iord, crdr, tlabel, doc |
| Origen pre | report, line, stmt, inpth, rfile, plabel, negating |
| Origen sub | cik, name, sic, countryba, stprba, cityba, zipba, bas1, bas2, baph, countryma, stprma, cityma, zipma, mas1, mas2, countryinc, stprinc, ein, former, changed, afs, fye, form, period, fy, fp, filed, accepted, prevrpt, detail, instance, nciks,aciks |

Cada fila de full_df representa un **valor financiero individual** reportado por una empresa, enriquecido con la definición del concepto contable, su posición en el estado financiero y los metadatos del reporte.

In [ ]:
num_tag_df = num_df.join(
    tag_df,
    on=["tag", "version"],
    how="left"
)

In [ ]:
num_tag_pre_df = num_tag_df.join(
    pre_df,
    on=["adsh", "tag", "version"],
    how="left"
)

In [ ]:
full_df = num_tag_pre_df.join(
    sub_df,
    on="adsh",
    how="left"
)

## 4. Análisis de Calidad de Datos

### Pain points del Dataset según la SEC

**A. Dimensiones duplicadas en `num.txt`**
Un mismo concepto (`tag`) puede aparecer varias veces para el mismo reporte (`adsh`) si la empresa lo reportó en diferentes *dimensiones* tales como consolidado vs. por segmento.
> **Riesgo:** Si se suma la columna `value` sin filtrar, se inflarán los números.
---

**B. Versiones de Tags (Custom Tags)**
Muchas empresas no usan los tags estándar de US-GAAP, sino que crean los suyos, como ejemplo, `version` empezará con el CIK de la empresa.
> **Riesgo:** Dificulta comparativas entre bancos.
---

**C. Enmiendas (Formularios 10-K/A)**
Si una empresa se equivocó, sube un reporte con `/A`.
> **Riesgo:** Se tendrán dos registros para el mismo periodo. Es importante decidir si preferimos quedarnos con el más reciente usando la columna `accepted`.

In [ ]:
# Dimensiones básicas
num_rows = full_df.count()
num_cols = len(full_df.columns)

print(f"--- Dimensiones del Dataset ---")
print(f"Registros totales: {num_rows:,}")
print(f"Columnas totales: {num_cols}")
print("-" * 30)

# Tipos de datos
full_df.printSchema()

--- Dimensiones del Dataset ---
Registros totales: 50,628,691
Columnas totales: 59
------------------------------
root
 |-- adsh: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- version: string (nullable = true)
 |-- ddate: string (nullable = true)
 |-- qtrs: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- segments: string (nullable = true)
 |-- coreg: string (nullable = true)
 |-- value: string (nullable = true)
 |-- footnote: string (nullable = true)
 |-- custom: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- datatype: string (nullable = true)
 |-- iord: string (nullable = true)
 |-- crdr: string (nullable = true)
 |-- tlabel: string (nullable = true)
 |-- doc: string (nullable = true)
 |-- report: string (nullable = true)
 |-- line: string (nullable = true)
 |-- stmt: string (nullable = true)
 |-- inpth: string (nullable = true)
 |-- rfile: string (nullable = true)
 |-- plabel: string (nullable = true)
 |-- negating: string (n

In [ ]:
# Conteo de nulos en una sola pasada de PySpark
total_rows = full_df.count()

null_count_exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in full_df.columns
]

null_counts_wide = full_df.select(*null_count_exprs)

# Convertimos de formato ancho a largo para mostrarlo como tabla
stack_expr = "stack({}, {}) as (columna, nulos)".format(
    len(full_df.columns),
    ", ".join([f"'{c}', `{c}`" for c in full_df.columns])
)

# Creación de la tabla final con porcentaje de nulos por columna
null_counts_table = (
    null_counts_wide
    .selectExpr(stack_expr)
    .withColumn(
        "pct_nulos",
        F.round((F.col("nulos") / F.lit(total_rows)) * 100, 4).cast("decimal(10,4)")
    )
    .orderBy(F.desc("nulos"), F.asc("columna"))
)

print("--- Conteo de nulos por columna ---")
null_counts_table.show(truncate=False)

--- Conteo de nulos por columna ---
+----------+--------+---------+
|columna   |nulos   |pct_nulos|
+----------+--------+---------+
|footnote  |50505319|99.7563  |
|coreg     |49541451|97.8525  |
|aciks     |47740082|94.2945  |
|bas2      |27132538|53.5912  |
|mas2      |27098655|53.5243  |
|changed   |19922506|39.3502  |
|former    |19922506|39.3502  |
|segments  |18317570|36.1802  |
|sic       |10068384|19.8867  |
|stprinc   |9513890 |18.7915  |
|crdr      |7236447 |14.2932  |
|stprba    |5176741 |10.2249  |
|stprma    |5060192 |9.9947   |
|countryinc|4933691 |9.7449   |
|fp        |3405224 |6.7259   |
|fy        |3404225 |6.7239   |
|value     |2185015 |4.3158   |
|countryma |700917  |1.3844   |
|countryba |643067  |1.2702   |
|afs       |514476  |1.0162   |
+----------+--------+---------+
only showing top 20 rows


In [ ]:
# Estadísticas descriptivas de las columnas numéricas (como 'value')
print("--- Resumen Estadístico (Columnas Numéricas) ---")
full_df.select("value").summary("count", "min", "25%", "50%", "75%", "max").show()

--- Resumen Estadístico (Columnas Numéricas) ---
+-------+------------------+
|summary|             value|
+-------+------------------+
|  count|          48443676|
|    min|           -0.0001|
|    25%|              0.01|
|    50%|         1781000.0|
|    75%|          4.4803E7|
|    max|9999999999999.0000|
+-------+------------------+



In [ ]:
# Calcular porcentaje de completitud (4 decimales)
total = full_df.count()
df_stats = full_df.select([
    F.round((F.count(c) / total) * 100, 4).cast("decimal(10,4)").alias(c)
    for c in full_df.columns
])

print("--- Porcentaje de datos presentes (Completitud) ---")
df_stats.show()

# Identificar registros con el campo 'value' nulo (que no nos sirven para análisis)
registros_sin_valor = full_df.filter(F.col("value").isNull()).count()
print(f"Alerta: Hay {registros_sin_valor:,} registros sin valor numérico.")

--- Porcentaje de datos presentes (Completitud) ---
+--------+--------+--------+--------+--------+--------+--------+------+-------+--------+--------+--------+--------+--------+-------+-------+-------+-------+-------+-------+-------+-------+-------+--------+--------+--------+-------+---------+-------+-------+-------+-------+-------+-------+---------+-------+-------+-------+-------+-------+----------+-------+--------+-------+-------+-------+--------+-------+--------+-------+-------+-------+--------+--------+--------+--------+--------+--------+------+
|    adsh|     tag| version|   ddate|    qtrs|     uom|segments| coreg|  value|footnote|  custom|abstract|datatype|    iord|   crdr| tlabel|    doc| report|   line|   stmt|  inpth|  rfile| plabel|negating|     cik|    name|    sic|countryba| stprba| cityba|  zipba|   bas1|   bas2|   baph|countryma| stprma| cityma|  zipma|   mas1|   mas2|countryinc|stprinc|     ein| former|changed|    afs|    wksi|    fye|    form| period|     fy|     fp|   f

In [ ]:
# Registros con al menos un valor nulo en cualquier columna
filas_con_nulo = full_df.filter(
    F.greatest(*[F.col(c).isNull().cast("int") for c in full_df.columns]) == 1
).count()

print(f"Registros con al menos un campo nulo: {filas_con_nulo:,}")
print(f"Porcentaje del total: {round(filas_con_nulo / total_rows * 100, 4):.4f}%")

Registros con al menos un campo nulo: 50,628,423
Porcentaje del total: 99.9995%


## 5. Caracterización de la Población

A continuación se presenta la tabla de caracterización de las 59 variables del dataset consolidado.
Para cada variable se incluye: tipo de dato, dominio (valores posibles), estadísticos descriptivos
(moda para categóricas, cuartiles para numéricas) y porcentaje de nulos.

Esta caracterización permite identificar los comportamientos habituales de la población y establecer el tipo y rangos de valores válidos para cada variable.

In [ ]:
cols_decimal = ["value"]
cols_bool = ["custom", "abstract", "inpth", "negating", "wksi", "prevrpt", "detail"]
cols_int = ["qtrs", "line", "fy", "report", "cik", "sic", "ein", "nciks"]
cols_date = ["ddate"]

# Ciclo para convertir las columnas a los tipos adecuados:
for col in cols_decimal:
    full_df = full_df.withColumn(col, F.col(col).cast(DecimalType(28,4)))

for col in cols_int:
    full_df = full_df.withColumn(col, F.col(col).cast(IntegerType()))

for col in cols_bool:
    full_df = full_df.withColumn(col, F.col(col).cast(BooleanType()))

for col in cols_date:
    full_df = full_df.withColumn(col, F.col(col).cast(DateType()))

In [ ]:
#  Función para guardar las caracterizaciones y no correr todo el proceso cada vez
def guardar_caracterizacion(df, base_path):
    path = os.path.join(base_path, "caracterizacion_cache.csv")
    df.to_csv(path, index=False)
    print(f"Cache guardado en {path}")

# Función para cargar la caracterización desde el cache y no correr todo el proceso cada vez
def cargar_caracterizacion(base_path):
    path = os.path.join(base_path, "caracterizacion_cache.csv")
    if not os.path.exists(path):
        print("No se encontró el cache. Ejecuta primero la caracterización.")
        return None
    df = pd.read_csv(path)
    print(f"Cargado desde cache: {len(df)} variables")
    return df

In [ ]:
total_registros = full_df.count()
resumen_final = []

CACHE_PATH = os.path.join(base_path, "caracterizacion_cache.csv")
cache_disponible = os.path.exists(CACHE_PATH)

if cache_disponible:
    df_visual = cargar_caracterizacion(base_path)
else:
    # Caracterización detallada por columna del dataset
    for col_name in full_df.columns:
        print(f"Procesando caracterización de {col_name}")

        dtype = full_df.schema[col_name].dataType
        dtype_str = dtype.simpleString()

        nulos_count = full_df.filter(F.col(col_name).isNull()).count()
        pct_nulos = (nulos_count / total_registros) * 100

        # Para análisis de dominio y estadísticos, consideramos solo filas no nulas
        df_clean = full_df.select(col_name).filter(F.col(col_name).isNotNull())

        dominio = "N/A"
        estadisticos = "N/A"

        # Solo analizamos dominio y estadísticos si hay datos no nulos
        if df_clean.count() > 0:

            if isinstance(dtype, (DecimalType, IntegerType)): #variables numéricas
                rango = df_clean.select(F.min(col_name), F.max(col_name)).first()
                dominio = f"{rango[0]} -> {rango[1]}"

                # Media y desviación estándar
                stats_row = df_clean.select(
                    F.mean(F.col(col_name).cast('double')).alias('mean'),
                    F.stddev(F.col(col_name).cast('double')).alias('stddev'),
                    F.count(col_name).alias('count_no_null')
                ).first()
                mean_val = stats_row['mean']
                std_val = stats_row['stddev']
                count_nn = stats_row['count_no_null']

                q = df_clean.stat.approxQuantile(col_name, [0.25, 0.50, 0.75], 0.01)
                estadisticos = (f"Count: {count_nn:,}, Mean: {mean_val:.2f}, Stddev: {std_val:.2f}, "
                            f"Q25: {q[0]:.2f}, Q50: {q[1]:.2f}, Q75: {q[2]:.2f}")

            else: #variables categóricas
                distintos = df_clean.select(col_name).distinct().limit(10).collect() #los primeros 10 valores únicos
                vals = [str(r[0]) for r in distintos]
                dominio = ", ".join(vals) + ("..." if len(vals) == 10 else "")

                moda_raw = df_clean.groupBy(col_name).count().orderBy(F.desc("count")).first()
                estadisticos = f"Moda: '{moda_raw[0]}' (Freq: {moda_raw[1]:,})"

        resumen_final.append(Row(
            Variable=col_name,
            Tipo_Dato=dtype_str,
            Dominio=dominio,
            Porcentaje_Nulos=f"{pct_nulos:.2f}%",
            Estadisticos=estadisticos
        ))

Procesando caracterización de adsh
Procesando caracterización de tag
Procesando caracterización de version
Procesando caracterización de ddate
Procesando caracterización de qtrs
Procesando caracterización de uom
Procesando caracterización de segments
Procesando caracterización de coreg
Procesando caracterización de value
Procesando caracterización de footnote
Procesando caracterización de custom
Procesando caracterización de abstract
Procesando caracterización de datatype
Procesando caracterización de iord
Procesando caracterización de crdr
Procesando caracterización de tlabel
Procesando caracterización de doc
Procesando caracterización de report
Procesando caracterización de line
Procesando caracterización de stmt
Procesando caracterización de inpth
Procesando caracterización de rfile
Procesando caracterización de plabel
Procesando caracterización de negating
Procesando caracterización de cik
Procesando caracterización de name
Procesando caracterización de sic
Procesando caracterizaci

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

# Convertimos el resumen a DataFrame para visualización
if not cache_disponible:
    df_visual = pd.DataFrame(resumen_final)
    df_visual.columns = ["Variable", "Tipo de Dato", "Dominio", "% Nulos", "Estadísticos"]

df_visual

,Variable,Tipo de Dato,Dominio,% Nulos,Estadísticos
0,adsh,string,"0001493152-23-042479, 0001628280-23-036378, 0001213900-23-084866, 0000055242-23-000051, 0001120970-23-000072, 0001410578-23-002402, 0000924901-23-000035, 0001628280-23-038221, 0001104659-23-116417, 0001654954-23-014359...",0.00%,"Moda: '0001193125-23-277031' (Freq: 55,852)"
1,tag,string,"WeightedAverageNumberOfDilutedSharesOutstanding, CostOfRevenue, OtherComprehensiveIncomeLossFinancialLiabilityFairValueOptionAfterTaxAndReclassificationAdjustment, AccretionOfClassAOrdinarySharesToAccretedValue, StockIssuedDuringWarrantsExercisedForCashValue, RepaymentsOfDebtAndCapitalLeaseObligations, FranchisorCosts, DecommissioningTrustAssetsAmount, InterestExpenseFederalHomeLoanBankAndFederalReserveBankAdvances, RepaymentsOfOtherDebt...",0.00%,"Moda: 'StockholdersEquity' (Freq: 3,894,858)"
2,version,string,"0001493152-23-042479, 0001493152-23-040957, 0000924805-23-000114, 0001020859-23-000115, 0000950170-23-065631, 0000924901-23-000035, 0001755755-23-000022, 0001628280-23-040545, 0001825079-23-000044, 0001069157-23-000114...",0.00%,"Moda: 'us-gaap/2024' (Freq: 18,205,875)"
3,ddate,date,N/A,100.00%,N/A
4,qtrs,int,0 -> 3604,0.00%,"Count: 50,628,691, Mean: 1.15, Stddev: 2.55, Q25: 0.00, Q50: 0.00, Q75: 2.00"
5,uom,string,"numberOfSegments, Contracts, NumberOfContracts, contract, Casino_Property, Horizen, INR, BND/VND, Units, ISK...",0.00%,"Moda: 'USD' (Freq: 42,917,994)"
6,segments,string,"InvestmentIdentifier=BioXcel Therapeutics, Inc., First Lien Term Loan 3;, InvestmentIdentifier=American Residential Services L.L.C. and Aragorn Parent Holdings LP, Series A preferred units;, InvestmentIdentifier=TI Intermediate Holdings, LLC, Senior Secured 4;, ClassOfWarrantOrRight=ImmediatelyExercisableWarrant;, InvestmentIdentifier=M C Asset Management ( Corporate), L L C, Senior Secured Loans Due 1/26/2024;, ConsolidatedEntities=ParentCompany;, ProductOrService=MixAndEquipmentRevenueFromFranchisees;, InvestmentIdentifier=Community Brands ParentCo, LLC, First lien senior secured loan;, SegmentConsolidationItems=UnallocatedAmounts;, FinancingReceivablePortfolioSegment=CommercialRealEstatePortfolioSegment;FinancingReceivableRecordedInvestmentByClassOfFinancingReceivable=OwnerOccupiedProperties;...",36.18%,"Moda: 'EquityComponents=CommonStock;' (Freq: 2,057,966)"
7,coreg,string,"DTEElectric, NewStemLtd, PeaceRiverMill, NationalInstitutesOfHealth, NetCoPartners, Hvpseventure, MacquarieUsTradingLlc, GarrisonCapitalEquityHoldingsIiLlc, VerdeCapitalLLC, AtpOilAndGasCorporation...",97.85%,"Moda: 'ProgressEnergy' (Freq: 18,436)"
8,value,"decimal(28,4)",-305780046000000.0000 -> 500000000000000000000.0000,4.32%,"Count: 48,443,676, Mean: 61949007310802.30, Stddev: 166055223844810784.00, Q25: 0.01, Q50: 1730000.00, Q75: 44000000.00"
9,footnote,string,"Dividends declared per common share during the nine months ended September 30, 2023 - $1.65 (2022 - $1.65)., Presented net of reclassification adjustments, which are not material in any period presented., Includes related-party amounts of $1,448 and $6,666 for the three and nine months ended September 30, 2023, respectively, and $1,742 and $6,027 for the three and nine months ended September 30, 2022, respectively, Investments without an interest rate are non-income producing., The interest rate on this loan is subject to 6 month EURIBOR, which as of December 31, 2022 was 2.69%., Dividends to shareholders are recorded in the period in which they are declared., The interest rate is the designated annual interest rate exclusive of any original issue discount, fees or end of term payment., Retroactively restated for the reverse recapitalization as described in Note 3., The underlying credit agreement or indenture contains a PIK provision, whereby the issuer has either the option or the obligation to make a portion of the interest payments with the issuance of additional investments. The PIK portion of the coupon is L+13.50%., Amount includes certain reclass

In [ ]:
if false: # Cambia a True para guardar el cache después de correr la caracterización
    guardar_caracterizacion(df_visual, base_path)

Cache guardado en C:\Users\getec\Documents\MNA\BigData\Proyecto\Entregable1\SEC Financial Statement Data Sets\SEC Financial Statement Data Sets\caracterizacion_cache.csv


## 6. Descripción de las Reglas de Particionamiento

### 6.1 Variables de Particionamiento Candidatas

Se identificaron **cuatro variables de caracterización** candidatas para particionar la base de datos D:

| Variable | Descripción | Valores posibles | Justificación |
|----------|-------------|-----------------|---------------|
| `fy` (Año Fiscal) | Año fiscal del reporte | 2012–2027 (concentrado en 2023–2025) | Partición natural para series de tiempo |
| `fp` (Periodo Fiscal) | Trimestre o año completo | FY, Q1, Q2, Q3 | Segmenta por estacionalidad |
| `form` (Tipo de Formulario) | Formulario SEC presentado | 10-K, 10-Q, 20-F, 40-F, S-1, ... | Estratifica por rigurosidad del reporte |
| `stmt` (Estado Financiero) | Tipo de estado financiero | BS, IS, CF, EQ, CI, SI, CP, UN | Separa por naturaleza contable |


### 6.2 Análisis Exploratorio: Particionamiento con 4 Variables

A continuación se muestra el análisis exploratorio con 4 variables (`fy`, `fp`, `form`, `stmt`)
que justifica la decisión de reducir a 2 variables.

In [ ]:
partition_vars = ["fy", "fp", "form", "stmt"]

total_n = full_df.count()

df_partitions = full_df.filter("fy IS NOT NULL AND fp IS NOT NULL AND form IS NOT NULL AND stmt IS NOT NULL") \
    .groupBy(*partition_vars) \
    .agg(F.count("*").alias("frecuencia_absoluta"))

df_partitions = df_partitions.withColumn(
    "probabilidad_ocurrencia",
    F.round((F.col("frecuencia_absoluta") / total_n) * 100, 5)
)

df_partitions = df_partitions.orderBy(F.desc("probabilidad_ocurrencia"))

In [ ]:
print(f"Número total de combinaciones (4 variables): {df_partitions.count()}")
print(f"Probabilidad máxima: {df_partitions.first()['probabilidad_ocurrencia']:.2f}%")
print("\nTop 10 combinaciones más frecuentes:")
df_partitions.show(10)

Número total de combinaciones (4 variables): 899
Probabilidad máxima: 2.27%

Top 10 combinaciones más frecuentes:
+----+---+----+----+-------------------+-----------------------+
|  fy| fp|form|stmt|frecuencia_absoluta|probabilidad_ocurrencia|
+----+---+----+----+-------------------+-----------------------+
|2025| Q3|10-Q|  BS|            1151569|                2.27454|
|2024| Q3|10-Q|  EQ|            1133748|                2.23934|
|2025| Q3|10-Q|  EQ|            1122410|                2.21694|
|2024| Q3|10-Q|  BS|            1112041|                2.19646|
|2025| Q2|10-Q|  BS|            1099901|                2.17249|
|2024| FY|10-K|  BS|            1061073|                2.09579|
|2024| Q2|10-Q|  BS|            1060273|                2.09421|
|2023| FY|10-K|  BS|            1025265|                2.02507|
|2025| Q1|10-Q|  BS|            1019621|                2.01392|
|2025| FY|10-K|  BS|            1005831|                1.98668|
+----+---+----+----+-------------------+-

### 6.2.1 Evaluación del Esquema de 4 Variables

Al combinar las 4 variables se generan **más de 500 combinaciones** de particionamiento, donde la combinación
más frecuente apenas alcanza el **2.27%** de probabilidad de ocurrencia. Esta atomización excesiva nos lleva a varios retos que se deben considerar como:
- Sobrecarga en gestión de metadatos
- Degradación de rendimiento en lectura/escritura de archivos Parquet
- Cuellos de botella por shuffling en operaciones distribuidas

### 6.3 Particionamiento Final: 2 Variables (`fy`, `fp`)

Con solo `fy` y `fp`, la cantidad de particiones se reduce drásticamente y las probabilidades de ocurrencia suben a niveles manejables, llevándonos a 6–10% para los estratos principales.

In [ ]:
partition_vars_final = ["fy", "fp"]

total_n = full_df.count()

df_partitions_final = full_df.filter("fy IS NOT NULL AND fp IS NOT NULL") \
    .groupBy(*partition_vars_final) \
    .agg(F.count("*").alias("frecuencia_absoluta"))

df_partitions_final = df_partitions_final.withColumn(
    "probabilidad_ocurrencia",
    F.round((F.col("frecuencia_absoluta") / total_n) * 100, 5)
)

df_partitions_final = df_partitions_final.orderBy(F.desc("probabilidad_ocurrencia"))

print(f"Número total de particiones: {df_partitions_final.count()}")
print("\nTabla completa de particiones (fy × fp):")
df_partitions_final.show(10)

Número total de particiones: 39

Tabla completa de particiones (fy × fp):
+----+---+-------------------+-----------------------+
|  fy| fp|frecuencia_absoluta|probabilidad_ocurrencia|
+----+---+-------------------+-----------------------+
|2024| FY|            5414351|               10.69423|
|2023| FY|            5178575|               10.22854|
|2025| Q3|            5149255|               10.17063|
|2025| Q2|            4930156|                9.73787|
|2024| Q3|            4809191|                9.49894|
|2025| FY|            4746697|                9.37551|
|2024| Q2|            4605726|                9.09707|
|2023| Q3|            4118571|                8.13486|
|2025| Q1|            3551373|                7.01455|
|2024| Q1|            3213032|                6.34627|
+----+---+-------------------+-----------------------+
only showing top 10 rows


### 6.3.1 Esquema de Particionamiento Final: 2 Variables (`fy`, `fp`)

Para mitigar los riesgos anteriores, se reduce el particionamiento a las **variables temporales** `fy` y `fp`.
Esto produce **39 combinaciones** con probabilidades de ocurrencia entre **6% y 10%** para los estratos principales,
logrando un equilibrio entre granularidad y eficiencia.


## 7. Extracción de Submuestras por Regla de Particionamiento

En esta sección se implementa el código que permite **filtrar y recuperar registros** de la base de datos D
que cumplan cada regla de particionamiento construida. Para cada combinación de `(fy, fp)`, se extrae la
submuestra correspondiente y se verifican sus características.

**NOTA:** En esta etapa solo se verifica el correcto funcionamiento del código. Las muestras obtenidas son de prueba.
En una etapa posterior, este código será utilizado para construir conjuntos de entrenamiento y prueba.

### 7.1 Función de Extracción de Particiones

In [ ]:
# Función para extraer una partición específica del DataFrame
def extraer_particion(df, fy_val, fp_val):
    particion = df.filter(
        (F.col("fy") == fy_val) & (F.col("fp") == fp_val)
    )
    return particion

### 7.2 Verificación: Extracción de Particiones Principales

In [ ]:
# Definir las particiones principales a verificar en nuestro análisis
particiones_prueba = [
    (2024, "FY"),
    (2023, "FY"),
    (2025, "Q3"),
    (2025, "Q2"),
    (2024, "Q3"),
    (2025, "FY"),
    (2024, "Q2"),
    (2023, "Q3"),
    (2025, "Q1"),
    (2024, "Q1"),
]

# Definición de títulos para la sección de verificación
print("=" * 80)
print("VERIFICACIÓN DE EXTRACCIÓN DE SUBMUESTRAS POR PARTICIÓN")
print("=" * 80)

resultados_verificacion = []

# Iteramos sobre cada combinación de partición, extraemos los registros y calculamos la probabilidad
for fy_val, fp_val in particiones_prueba:
    particion = extraer_particion(full_df, fy_val, fp_val)
    n_registros = particion.count()
    prob = round((n_registros / total_n) * 100, 4)

    # Guardamos los resultados en una lista para mostrar al final
    resultados_verificacion.append(Row(
        fy=fy_val, fp=fp_val,
        registros=n_registros,
        probabilidad_pct=prob
    ))

    # Mostramos un resumen de cada partición verificada
    print(f"\nPartición fy={fy_val}, fp={fp_val}: {n_registros:,} registros ({prob}%)")

# Resumen en tabla
print("\n" + "=" * 80)
print("RESUMEN DE VERIFICACIÓN")
print("=" * 80)

# Debido a problemas con Pyspark con el JVM, mostramos los resultados con pandas en lugar de PySpark
#spark.createDataFrame(resultados_verificacion).show()
pd.DataFrame(resultados_verificacion)

VERIFICACIÓN DE EXTRACCIÓN DE SUBMUESTRAS POR PARTICIÓN

Partición fy=2024, fp=FY: 5,414,351 registros (10.6942%)

Partición fy=2023, fp=FY: 5,178,575 registros (10.2285%)

Partición fy=2025, fp=Q3: 5,149,255 registros (10.1706%)

Partición fy=2025, fp=Q2: 4,930,156 registros (9.7379%)

Partición fy=2024, fp=Q3: 4,809,191 registros (9.4989%)

Partición fy=2025, fp=FY: 4,746,697 registros (9.3755%)

Partición fy=2024, fp=Q2: 4,605,726 registros (9.0971%)

Partición fy=2023, fp=Q3: 4,118,571 registros (8.1349%)

Partición fy=2025, fp=Q1: 3,551,373 registros (7.0145%)

Partición fy=2024, fp=Q1: 3,213,032 registros (6.3463%)

RESUMEN DE VERIFICACIÓN


,0,1,2,3
0,2024,FY,5414351,10.6942
1,2023,FY,5178575,10.2285
2,2025,Q3,5149255,10.1706
3,2025,Q2,4930156,9.7379
4,2024,Q3,4809191,9.4989
5,2025,FY,4746697,9.3755
6,2024,Q2,4605726,9.0971
7,2023,Q3,4118571,8.1349
8,2025,Q1,3551373,7.0145
9,2024,Q1,3213032,6.3463


### 7.3 Ejemplo: Contenido de una Partición

In [ ]:
# Ejemplo detallado: Partición fy=2024, fp=FY (la más grande)
print("Muestra de registros de la partición fy=2024, fp=FY:")
print("-" * 80)

particion_ejemplo = extraer_particion(full_df, 2024, "FY")

particion_ejemplo.select(
    "name", "form", "fy", "fp", "stmt", "tag", "plabel", "value", "uom"
).show(15, truncate=40)

Muestra de registros de la partición fy=2024, fp=FY:
--------------------------------------------------------------------------------
+------------------------------+----+----+---+----+----------------------------------------+----------------------------------------+----------------+---+
|                          name|form|  fy| fp|stmt|                                     tag|                                  plabel|           value|uom|
+------------------------------+----+----+---+----+----------------------------------------+----------------------------------------+----------------+---+
|    ADVANCED MICRO DEVICES INC|10-K|2024| FY|  IS|CostOfGoodsAndServiceExcludingDepreci...|                           Cost of sales|11550000000.0000|USD|
|    ADVANCED MICRO DEVICES INC|10-K|2024| FY|  IS|CostOfGoodsAndServiceExcludingDepreci...|                           Cost of sales|11278000000.0000|USD|
|    ADVANCED MICRO DEVICES INC|10-K|2024| FY|  IS|CostOfGoodsAndServiceExcludingDepreci...

### 7.4 Estadísticos Descriptivos por Partición

In [ ]:
# Estadísticos del campo 'value' para la partición fy=2024, fp=FY
print("Estadísticos de 'value' para partición fy=2024, fp=FY:")
particion_ejemplo.select("value").summary("count", "min", "25%", "50%", "75%", "max").show()

# Distribución por tipo de formulario dentro de la partición
print("Distribución por formulario (form) dentro de la partición:")
particion_ejemplo.groupBy("form").count().orderBy(F.desc("count")).show(10)

# Distribución por estado financiero dentro de la partición
print("Distribución por estado financiero (stmt) dentro de la partición:")
particion_ejemplo.groupBy("stmt").count().orderBy(F.desc("count")).show(10)

Estadísticos de 'value' para partición fy=2024, fp=FY:
+-------+--------------------+
|summary|               value|
+-------+--------------------+
|  count|             5214447|
|    min|-267792169000000....|
|    25%|              0.0316|
|    50%|           3304000.0|
|    75%|            84035000|
|    max|1271695074000000....|
+-------+--------------------+

Distribución por formulario (form) dentro de la partición:
+------+-------+
|  form|  count|
+------+-------+
|  10-K|4203951|
|  20-F| 904670|
|10-K/A| 132600|
|  40-F|  89908|
|20-F/A|  38172|
|   6-K|  21179|
| 10-KT|  11712|
|POS AM|   2252|
|   S-1|   1805|
| S-1/A|   1535|
+------+-------+
only showing top 10 rows
Distribución por estado financiero (stmt) dentro de la partición:
+----+-------+
|stmt|  count|
+----+-------+
|  BS|1427977|
|  IS|1009766|
|  EQ| 992844|
|  CF| 953793|
|  SI| 606913|
|  UN| 283699|
|  CI| 139047|
|NULL|    312|
+----+-------+



### 7.5 Extracción de Todas las Particiones

El siguiente código itera sobre todas las combinaciones de particionamiento identificadas y extrae
la submuestra correspondiente, verificando que la suma de registros de todas las particiones sea
consistente con el total del dataset.

In [ ]:
# Recopilar todas las combinaciones de partición
combinaciones = df_partitions_final.select("fy", "fp").collect()

print(f"Total de combinaciones de partición: {len(combinaciones)}")
print("=" * 80)

suma_registros = 0

# Iteramos sobre cada combinación de partición, extraemos los registros y sumamos el total para verificar
# que coincide con el total del dataset
for row in combinaciones:
    fy_val = row["fy"]
    fp_val = row["fp"]

    # Extraemos la partición específica y contamos sus registros
    particion = extraer_particion(full_df, fy_val, fp_val)
    n = particion.count()
    suma_registros += n
    print(f"  P(fy={fy_val}, fp={fp_val}): {n:>10,} registros")

# Registros con fy o fp nulo (no pertenecen a ninguna partición)
nulos_particion = full_df.filter("fy IS NULL OR fp IS NULL").count()

# Presentamos el resumen final de la verificación
print("=" * 80)
print(f"Suma de registros en particiones:     {suma_registros:,}")
print(f"Registros sin partición (fy/fp nulo): {nulos_particion:,}")
print(f"Total en dataset:                     {total_n:,}")
print(f"Verificación: {suma_registros} + {nulos_particion} = {suma_registros + nulos_particion} == {total_n} --> {'Correcto' if suma_registros + nulos_particion == total_n else 'Error'}")

Total de combinaciones de partición: 39
  P(fy=2024, fp=FY):  5,414,351 registros
  P(fy=2023, fp=FY):  5,178,575 registros
  P(fy=2025, fp=Q3):  5,149,255 registros
  P(fy=2025, fp=Q2):  4,930,156 registros
  P(fy=2024, fp=Q3):  4,809,191 registros
  P(fy=2025, fp=FY):  4,746,697 registros
  P(fy=2024, fp=Q2):  4,605,726 registros
  P(fy=2023, fp=Q3):  4,118,571 registros
  P(fy=2025, fp=Q1):  3,551,373 registros
  P(fy=2024, fp=Q1):  3,213,032 registros
  P(fy=2026, fp=Q1):    459,576 registros
  P(fy=2026, fp=Q2):    387,123 registros
  P(fy=2026, fp=Q3):    208,294 registros
  P(fy=2023, fp=Q2):    182,624 registros
  P(fy=2022, fp=FY):    117,250 registros
  P(fy=2023, fp=Q1):     61,212 registros
  P(fy=2026, fp=FY):     32,707 registros
  P(fy=2022, fp=Q3):     22,774 registros
  P(fy=2022, fp=Q2):      9,651 registros
  P(fy=2021, fp=FY):      7,290 registros
  P(fy=2022, fp=Q1):      5,604 registros
  P(fy=2021, fp=Q3):      1,902 registros
  P(fy=2021, fp=Q2):      1,487 regi

### 7.6 Ejemplo: Partición Pequeña

In [ ]:
# Ejemplo con una partición pequeña para verificar que también funciona
particion_peq = extraer_particion(full_df, 2022, "Q1")
n_peq = particion_peq.count()

print(f"Partición fy=2022, fp=Q1: {n_peq:,} registros")
print("\nMuestra de registros:")
particion_peq.select(
    "name", "form", "fy", "fp", "stmt", "tag", "value", "uom"
).show(10, truncate=35)

Partición fy=2022, fp=Q1: 5,604 registros

Muestra de registros:
+--------------------------+------+----+---+----+----------------------------+-------------+---+
|                      name|  form|  fy| fp|stmt|                         tag|        value|uom|
+--------------------------+------+----+---+----+----------------------------+-------------+---+
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  BS|AccountsReceivableNetCurrent|37869000.0000|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  BS|AccountsReceivableNetCurrent| -323000.0000|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  BS|AccountsReceivableNetCurrent|38424000.0000|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  BS|AccountsReceivableNetCurrent|38192000.0000|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  IS|       EarningsPerShareBasic|      -0.0400|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  IS|       EarningsPerShareBasic|       0.0100|USD|
|VINTAGE WINE ESTATES, INC.|10-Q/A|2022| Q1|  IS|       Earnin

## 8. Proceso de Muestreo

### 8.1 Técnica Seleccionada: Muestreo Estratificado Distribuido

Se emplea un **muestreo estratificado** por `fy` y `fp`, complementado con selección aleatoria
reproducible (semilla fija) dentro de cada estrato. Esta técnica es adecuada porque:

- Permite procesamiento paralelo en PySpark sobre grandes volúmenes
- Mantiene la trazabilidad y reproducibilidad de la selección
- Reduce sesgo al respetar la estructura temporal de los datos

### 8.2 Criterios de Muestreo

El tamaño de la muestra se determinará mediante el método de muestreo estratificado.

En primer lugar, se calculará el tamaño total de la muestra utilizando la fórmula de Yamane:

$${n} = \frac{N}{1+N({e^{2}})}$$

donde:

𝑛 = tamaño de la muestra

𝑁 = tamaño de la población

𝑒 = nivel de error permitido (en este caso, 1%)

Este cálculo se realizará con un nivel de confianza del 95% y un error máximo del 5%.

Posteriormente, el tamaño de la muestra se distribuirá entre los distintos estratos mediante la fórmula de asignación proporcional:

$${n_h} = \frac{N_h}{N}  n$$

donde:

${𝑛_ℎ}$ = tamaño de la muestra para el estrato ℎ

${N_ℎ}$ = tamaño de la población del estrato ℎ

𝑁 = tamaño total de la población

𝑛 = tamaño total de la muestra calculado con la fórmula de Yamane.

De esta manera, cada estrato estará representado en la muestra de forma proporcional a su peso dentro de la población total, garantizando mayor precisión y representatividad en los resultados del estudio.

In [ ]:
SEED = 42
E = 0.01 #e
sample_size = int(round(total_n/(1+total_n*E**2))) #n

# Función para muestrear una partición según su tamaño, esto es estratificado adaptativo
def muestrear_particion(df, fy_val, fp_val):

    # Utilizamos la función de extracción para obtener la partición específica
    particion = extraer_particion(df, fy_val, fp_val)
    n = particion.count() #Nh
    stratum_sample_size = int(round((n/total_n)*sample_size)) #nh

    # Si la partición está vacía, devolvemos un DataFrame vacío y una fracción de 1
    if n == 0:
        return particion, n, 0, 1.0

    # Aplicamos la lógica de muestreo según el tamaño de la partición
    muestra = particion.orderBy(F.rand(seed=SEED)).limit(stratum_sample_size)
    n_muestra = muestra.count()
    fraccion = n_muestra/n
    return muestra, n, n_muestra, fraccion

print(f"{'Tamaño final de la muestra:':} {sample_size:,}")

Tamaño final de la muestra: 9,998


### 8.3 Verificación del Muestreo Estratificado

In [ ]:
print("VERIFICACIÓN DEL MUESTREO ESTRATIFICADO")
print("=" * 80)
print(f"{'fy':<6} {'fp':<4} {'N original':>12} {'N muestra':>12} {'Fracción':>10}")
print("-" * 80)

total_muestra = 0

# Iteramos sobre las combinaciones de partición y aplicamos el muestreo, mostrando los resultados para las 15 particiones principales
for row in combinaciones[:15]:
    fy_val = row["fy"]
    fp_val = row["fp"]

    # Aplicamos la función de muestreo a cada partición y obtenemos el número original,
    # el número en la muestra y la fracción representada
    muestra, n_orig, n_muestra, fraccion = muestrear_particion(full_df, fy_val, fp_val)
    total_muestra += n_muestra

    print(f"{fy_val:<6} {fp_val:<4} {n_orig:>12,} {n_muestra:>12,} {(fraccion * 100):>10.2f}%")

print("-" * 80)
print(f"{'Total muestra (15 principales):':<34} {total_muestra:>12,}")

VERIFICACIÓN DEL MUESTREO ESTRATIFICADO
fy     fp     N original    N muestra   Fracción
--------------------------------------------------------------------------------
2024   FY      5,414,351        1,069       0.02%
2023   FY      5,178,575        1,023       0.02%
2025   Q3      5,149,255        1,017       0.02%
2025   Q2      4,930,156          974       0.02%
2024   Q3      4,809,191          950       0.02%
2025   FY      4,746,697          937       0.02%
2024   Q2      4,605,726          910       0.02%
2023   Q3      4,118,571          813       0.02%
2025   Q1      3,551,373          701       0.02%
2024   Q1      3,213,032          634       0.02%
2026   Q1        459,576           91       0.02%
2026   Q2        387,123           76       0.02%
2026   Q3        208,294           41       0.02%
2023   Q2        182,624           36       0.02%
2022   FY        117,250           23       0.02%
--------------------------------------------------------------------------------